In [ ]:
# main.py
import os
import json
from typing import List, Dict, Any
import numpy as np
from neo4j import GraphDatabase, basic_auth

# --- LLMs ---
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
from openai import OpenAI

# =========================
# ENV / CONFIG
# =========================
URI = os.environ.get("NEO4J_URI")          # change if Aura
USER = os.environ.get("NEO4J_USERNAME")
PASSWORD = os.environ.get("NEO4J_PASSWORD")
DB = os.environ.get("NEO4J_DATABASE")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]  # must be set
EMBED_MODEL = "text-embedding-3-small"         # or text-embedding-3-large

# =========================
# OPENAI / NEO4J CLIENTS
# =========================
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)
oa_client = OpenAI(api_key=OPENAI_API_KEY)
driver = GraphDatabase.driver(URI, auth=basic_auth(USER, PASSWORD))

def verify():
    print("[INFO] Connecting to:", URI)
    print("[INFO] DB:", DB)
    print("[INFO] USER:", USER)
    driver.verify_connectivity()
    print("[INFO] Connected. Server info:", driver.get_server_info())

# =========================
# LLM: TOPIC EXTRACTION
# =========================
TOPIC_SYS_PROMPT = """You extract the single most important keyword/topic that the paragraph is about.
Respond with ONLY the topic text, no punctuation or quotes. Keep it concise (max 8 words).
Examples:
- "The fourth season of Chicago Fire..." -> Chicago Fire Season 4
- "Apple unveiled the iPhone 16 Pro..." -> iPhone 16 Pro
"""

def extract_main_topic(llm: ChatOpenAI, paragraph: str) -> str:
    resp = llm.invoke([
        SystemMessage(content=TOPIC_SYS_PROMPT),
        HumanMessage(content=paragraph)
    ])
    topic = (resp.content or "").strip()
    return topic.replace('"', '').replace("'", "").strip()

# =========================
# EMBEDDINGS (BATCHED & SAFE)
# =========================
BATCH_SIZE = 256
MAX_NAME_CHARS = 200
MAX_CANDIDATES = 20000
PREFILTER_MIN_MATCH = 1  # # topic tokens required in name

def _sanitize_text(s: str) -> str:
    if not s: return " "
    s = s.strip()
    return s[:MAX_NAME_CHARS] if len(s) > MAX_NAME_CHARS else s

def _embed_batch(oa_client: OpenAI, texts: List[str]) -> np.ndarray:
    texts = [_sanitize_text(t) for t in texts]
    resp = oa_client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vecs / norms

def embed_texts_batched(oa_client: OpenAI, texts: List[str], batch_size: int = BATCH_SIZE) -> np.ndarray:
    if not texts:
        return np.zeros((0, 1536), dtype=np.float32)
    out = []
    for i in range(0, len(texts), batch_size):
        out.append(_embed_batch(oa_client, texts[i:i+batch_size]))
    return np.vstack(out)

def cosine_sim_matrix(vec_query: np.ndarray, vecs: np.ndarray) -> np.ndarray:
    return vecs @ vec_query

# =========================
# CANDIDATE FETCH (with cheap prefilter)
# =========================
def prettify_name(name_id_like: str) -> str:
    if not name_id_like:
        return ""
    x = name_id_like
    if x.startswith("val_"):
        x = x[4:]
    return x.replace("_", " ").strip()

# Generic all-nodes fallback (kept for safety)
GET_NODE_NAMES_CYPHER_ALL = """
MATCH (n)
OPTIONAL MATCH (n)-[:HAS_NAME]->(v:value_text)
WITH n, v
RETURN
  coalesce(n.id, elementId(n)) AS id,
  coalesce(v.id, n.id, elementId(n)) AS name_id_like,
  labels(n) AS graph_labels
"""

# Cheap token prefilter directly in Cypher (case-insensitive CONTAINS on id/name)
def fetch_nodes_prefiltered(topic: str) -> List[Dict[str, Any]]:
    tokens = [t for t in topic.lower().replace("-", " ").split() if len(t) >= 3]
    if not tokens:
        # No useful tokens → return all nodes (beware size)
        with driver.session(database=DB) as session:
            rows = session.run(GET_NODE_NAMES_CYPHER_ALL).data()
    else:
        # Build a WHERE with OR of tokens against id and name_id_like
        # We’ll MATCH all, but filter in WHERE
        where_clauses = []
        params = {}
        for idx, tok in enumerate(tokens):
            p = f"tok{idx}"
            params[p] = tok
            where_clauses.append(f"toLower(coalesce(v.id, n.id, elementId(n))) CONTAINS ${p}")
            where_clauses.append(f"toLower(coalesce(n.id, elementId(n))) CONTAINS ${p}")
        where = " OR ".join(where_clauses)
        cypher = f"""
        MATCH (n)
        OPTIONAL MATCH (n)-[:HAS_NAME]->(v:value_text)
        WITH n, v
        WHERE {where}
        RETURN
          coalesce(n.id, elementId(n)) AS id,
          coalesce(v.id, n.id, elementId(n)) AS name_id_like,
          labels(n) AS graph_labels
        LIMIT {MAX_CANDIDATES}
        """
        with driver.session(database=DB) as session:
            rows = session.run(cypher, params).data()

    results = []
    for r in rows:
        node_id = r.get("id")
        raw_name = r.get("name_id_like") or node_id
        results.append({
            "id": node_id,
            "raw_name": raw_name,
            "name": prettify_name(raw_name) or (node_id or ""),
            "graph_labels": r.get("graph_labels", []),
        })
    return results

def _cheap_prefilter_candidates_py(topic: str, candidates: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if not candidates or not topic:
        return candidates
    topic_tokens = [t for t in topic.lower().replace("-", " ").split() if len(t) >= 3]
    if not topic_tokens:
        return candidates
    kept = []
    for c in candidates:
        name = (c.get("name") or "").lower()
        hits = sum(1 for t in topic_tokens if t in name)
        if hits >= PREFILTER_MIN_MATCH:
            kept.append(c)
    return kept if len(kept) >= 50 else candidates

def find_most_similar_node(oa_client: OpenAI, topic: str, candidates: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not candidates:
        return {}
    # Optional Python prefilter as a second pass
    pool = _cheap_prefilter_candidates_py(topic, candidates)
    if len(pool) > MAX_CANDIDATES:
        pool = pool[:MAX_CANDIDATES]

    # Embed query once
    q_vec = _embed_batch(oa_client, [topic])[0]  # (D,)

    best_idx = -1
    best_sim = -1.0
    names = [c["name"] for c in pool]

    for i in range(0, len(names), BATCH_SIZE):
        vecs = _embed_batch(oa_client, names[i:i+BATCH_SIZE])  # (B, D)
        sims = vecs @ q_vec
        j = int(np.argmax(sims))
        if sims[j] > best_sim:
            best_sim = float(sims[j])
            best_idx = i + j

    if best_idx >= 0:
        best = pool[best_idx].copy()
        best["similarity"] = best_sim
        return best
    return {}

# =========================
# NEIGHBORS (with fallback to elementId)
# =========================
GET_NEIGHBORS_BY_ID = """
MATCH (center {id: $id})
OPTIONAL MATCH (center)-[r]-(nbr)
RETURN
  center.id AS center_id,
  type(r)   AS rel_type,
  CASE
    WHEN r IS NULL THEN NULL
    WHEN startNode(r) = center THEN 'OUT'
    ELSE 'IN'
  END AS direction,
  coalesce(nbr.id, elementId(nbr)) AS neighbor_id,
  labels(nbr) AS neighbor_graph_labels
ORDER BY rel_type, neighbor_id
"""

GET_NEIGHBORS_BY_COALESCE = """
MATCH (center)
WHERE coalesce(center.id, elementId(center)) = $id
OPTIONAL MATCH (center)-[r]-(nbr)
RETURN
  coalesce(center.id, elementId(center)) AS center_id,
  type(r)   AS rel_type,
  CASE
    WHEN r IS NULL THEN NULL
    WHEN startNode(r) = center THEN 'OUT'
    ELSE 'IN'
  END AS direction,
  coalesce(nbr.id, elementId(nbr)) AS neighbor_id,
  labels(nbr) AS neighbor_graph_labels
ORDER BY rel_type, neighbor_id
"""

def fetch_connections(node_id: str) -> List[Dict[str, Any]]:
    with driver.session(database=DB) as session:
        rows = session.run(GET_NEIGHBORS_BY_ID, {"id": node_id}).data()
    rows = [r for r in rows if r.get("rel_type") is not None]
    if rows:
        return rows
    # fallback if node had no `id` property and we were given elementId
    with driver.session(database=DB) as session:
        rows2 = session.run(GET_NEIGHBORS_BY_COALESCE, {"id": node_id}).data()
    return [r for r in rows2 if r.get("rel_type") is not None]

# =========================
# MAIN (demo run)
# =========================
if __name__ == "__main__":
    verify()

    paragraph = "what was the name of atom bomb dropped by usa on hiroshima"
    topic = extract_main_topic(llm, paragraph)
    print(f"[INFO] Extracted topic: {topic}")

    # Pull candidates with Cypher prefilter to avoid embedding the whole graph
    candidates = fetch_nodes_prefiltered(topic)
    if not candidates:
        print("[WARN] Your graph seems empty (no nodes returned).")
        exit(0)

    best = find_most_similar_node(oa_client, topic, candidates)
    if not best:
        print("[WARN] Could not determine the most similar node.")
        exit(0)

    neighbors = fetch_connections(best["id"])

    # Print summary
    print("\n=== Topic Extracted ===")
    print(topic)
    print("\n=== Best Match Node ===")
    print(f"ID: {best['id']}")
    print(f"Name: {best['name']}")
    print(f"Similarity: {best['similarity']:.4f}")
    print(f"Labels (graph): {best.get('graph_labels')}")
    print("\n=== Connections (all neighbors) ===")
    if not neighbors:
        print("[INFO] No connections found.")
    else:
        for r in neighbors:
            if r["direction"] == "IN":
                print(f"{r['neighbor_id']} -[{r['rel_type']}]-> {best['id']}")
            else:
                print(f"{best['id']} -[{r['rel_type']}]-> {r['neighbor_id']}")


/tmp/ipykernel_3072999/1558724107.py:27: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)


[INFO] Connecting to: neo4j+s://62b9e173.databases.neo4j.io
[INFO] DB: neo4j
[INFO] USER: neo4j
[INFO] Connected. Server info: <neo4j.api.ServerInfo object at 0x718bb9b44fa0>
[INFO] Extracted topic: Little Boy
[WARN] Your graph seems empty (no nodes returned).
[WARN] Could not determine the most similar node.


KeyError: 'id'

: 